<a href="https://colab.research.google.com/github/stevenolanecon/7002LBSAI/blob/main/notebooks/week2_instructor_sampling_distribution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 – Instructor demo: how a sampling distribution gets sharper

**For live use in the workshop, not a student handout.** This runs the Week 2 slide's own "Task" live: `samples.csv` holds 1,000 simulated replays of a class like ours – same structure as our real Week 1 data (15 students, 10 trials each), regenerated 1,000 times with the exact simulation code from `week2.qmd`.

The slide's own interactive chart already lets the class drag a slider through this. This notebook is the same idea run as real, visible code – useful if you want to run it live instead of (or alongside) the slide, or answer a spontaneous "what if" question the slide can't anticipate.

**The story:** with only 15 points (one simulated class), the histogram of student-level means is jagged and could look like almost anything. Pool in more replicate classes – 10, then 100, then all 1,000 – and the same histogram resolves into a smooth, stable shape. Nothing about the underlying population changed; only how much of it we're able to see changed.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

samples_path = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/samples.csv"
df = pd.read_csv(samples_path)
print(df.shape)
df.head()

`df` has one row per (sample, student, trial) – 1,000 samples x 15 students x 10 trials = 150,000 rows. Each "sample" is one full simulated replay of our 15-person class.

## Step 1 – one simulated class (sample 1 only)

In [ ]:
sample_1 = df[df['sample'] == 1]
student_means_1 = sample_1.groupby('student_name')['reaction_time'].mean()
print('n =', len(student_means_1))

plt.figure(figsize=(6, 4))
plt.hist(student_means_1, bins=8, color='#457b9d', edgecolor='white')
plt.xlabel('Mean reaction time (ms)')
plt.ylabel('Number of students')
plt.title('Sample 1 only – 15 student means')
plt.show()

**Talking point:** only 15 points – this histogram could look almost any shape by chance. Don't read too much into its exact form yet.

## Step 2 – pool in more replicate classes

In [ ]:
def student_means_up_to(k):
    sub = df[df['sample'] <= k]
    return sub.groupby(['sample', 'student_name'])['reaction_time'].mean()

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharex=True, sharey=False)
for ax, k in zip(axes, [1, 10, 100, 1000]):
    means_k = student_means_up_to(k)
    ax.hist(means_k, bins=30, color='#457b9d', edgecolor='white')
    ax.set_title(f'k = {k} sample(s)\nn = {len(means_k)}')
    ax.set_xlabel('Mean reaction time (ms)')
axes[0].set_ylabel('Count')
plt.tight_layout()
plt.show()

**Talking point:** the shape doesn't fundamentally change as k grows – it was always roughly this shape. What changes is how clearly we can *see* it. At k=1 we're looking through a keyhole; at k=1,000 the picture is fully resolved.

## Step 3 – mean and SD of the full pooled distribution

In [ ]:
means_1000 = student_means_up_to(1000)
print('Pooling all 1,000 samples (n =', len(means_1000), 'student-means):')
print('  mean:', round(means_1000.mean(), 1), 'ms')
print('  SD:  ', round(means_1000.std(), 1), 'ms')

print()
print('For comparison, sample 1 alone (n = 15):')
print('  mean:', round(student_means_1.mean(), 1), 'ms')
print('  SD:  ', round(student_means_1.std(), 1), 'ms')

**Answering the slide's questions:**

- *What is the mean of those student means? What is the SD?* – with all 1,000 samples pooled, roughly 281 ms and 49 ms (your exact numbers will differ slightly by run/seed, but should land close to this).
- *You are not plotting raw reaction times. What are you plotting the distribution of?* – the distribution of **a student's mean reaction time across many hypothetical replays of the same class**. It mixes two things: genuine between-student variation (some people are just faster) and the sampling noise left over from averaging only 10 trials per student per replay. It is *not* the distribution of individual trials (that would be wider and centred the same, but without the trial-level noise averaged out) – it's a preview of the idea of a **sampling distribution**, which the rest of this week builds on formally.